## Desafio D — Sistema de Registro de Preços

### Problema

Quais características diferenciam contratações realizadas com e sem Sistema de Registro de Preços?

### Possíveis perguntas

- O SRP é mais frequente em determinados tipos de contratação?
- Existem diferenças nos valores das contratações?
- Determinados órgãos utilizam SRP proporcionalmente mais do que outros?

### Variável de interesse

Quando disponível:

```text
srp
```

### Possíveis análises

- proporções;
- tabelas cruzadas;
- comparação de valores;
- teste qui-quadrado.


documentação API: https://dadosabertos.compras.gov.br/swagger-ui/index.html


In [1]:
#!pip install requests pandas matplotlib -q
#!pip install pyarrow

In [2]:
import requests
import pandas as pd
import matplotlib.pyplot as plt
import time

In [3]:
BASE_URL = "https://dadosabertos.compras.gov.br"

ENDPOINT_CONTRATACOES = "/modulo-contratacoes/1_consultarContratacoes_PNCP_14133"

url = BASE_URL + ENDPOINT_CONTRATACOES

print(url)

https://dadosabertos.compras.gov.br/modulo-contratacoes/1_consultarContratacoes_PNCP_14133


In [4]:
def extrair_registros(json_resposta):
    if isinstance(json_resposta, list):
        return json_resposta

    if not isinstance(json_resposta, dict):
        return []

    for chave in ["resultado", "resultados", "data", "content"]:
        if chave in json_resposta and isinstance(json_resposta[chave], list):
            return json_resposta[chave]

    return []



# Pegando dados 2024

In [5]:
modalidades = [5, 6] # Pegar todas as modalidades de licitação, mas para fins de teste, vamos pegar apenas as principais.
todos_registros = []

In [7]:
for modalidade in modalidades:
    pagina = 1
    total_modalidade = 0

    while True:
        params = {
            "pagina": pagina,
            "tamanhoPagina": 500,
            "dataPublicacaoPncpInicial": "2024-01-01",
            "dataPublicacaoPncpFinal": "2024-12-31",
            "codigoModalidade": modalidade,
            "unidadeOrgaoUfSigla": "SP"
        }

        resposta = requests.get(url, params=params, timeout=60)

        if resposta.status_code == 429:
            espera = 5
            print(f"Modalidade {modalidade}, página {pagina}: 429, aguardando {espera}s e tentando de novo...")
            time.sleep(espera)
            continue  # tenta a mesma página de novo, sem avançar

        if resposta.status_code != 200:
            print(f"Modalidade {modalidade}, página {pagina}: erro {resposta.status_code}")
            break

        dados = resposta.json()
        registros = extrair_registros(dados)

        if not registros:
            break

        todos_registros.extend(registros)
        total_modalidade += len(registros)

        if len(registros) < 500:
            break

        pagina += 1
        time.sleep(0.2)  # pausa maior entre páginas

    print(f"Modalidade {modalidade}: {total_modalidade} registros coletados.")
    time.sleep(3)  # pausa entre modalidades, para o servidor "esfriar"

Modalidade 5: 33327 registros coletados.
Modalidade 6: 36074 registros coletados.


In [8]:
# print("JSON retornado:", registros)
df_24 = pd.json_normalize(todos_registros)
df_24.head()

,idCompra,numeroControlePNCP,anoCompraPncp,sequencialCompraPncp,orgaoEntidadeCnpj,orgaoSubrogadoCnpj,codigoOrgao,orgaoEntidadeRazaoSocial,orgaoSubrogadoRazaoSocial,orgaoEntidadeEsferaId,...,tipoInstrumentoConvocatorioNome,modoDisputaNomePncp,valorTotalEstimado,valorTotalHomologado,dataInclusaoPncp,dataAtualizacaoPncp,dataPublicacaoPncp,dataAberturaPropostaPncp,dataEncerramentoPropostaPncp,contratacaoExcluida
0,17013105000172023,00394460000141-1-001631/2023,2023,1631,00394460000141,NaN,45178,MINISTERIO DA FAZENDA,NaN,F,...,Edital,Aberto-Fechado,614963.83,343793.50,2024-01-02T07:00:02,2024-01-02T07:00:02,2024-01-02T07:00:02,2024-01-02T09:00:00,2024-01-22T10:00:00,False
1,12006005000362023,00394429000100-1-002260/2023,2023,2260,00394429000100,NaN,43849,COMANDO DA AERONAUTICA,NaN,F,...,Edital,Aberto,286011.82,270000.00,2024-01-02T07:00:07,2024-01-02T07:00:07,2024-01-02T07:00:07,2024-01-02T08:00:00,2024-01-17T09:00:00,False
2,51017805000112023,29979036000140-1-000084/2023,2023,84,29979036000140,NaN,26661,INSTITUTO NACIONAL DO SEGURO SOCIAL,NaN,F,...,Edital,Aberto,232490.96,NaN,2024-01-02T07:00:09,2024-02-28T07:09:02,2024-01-02T07:00:09,2024-01-02T09:00:00,2024-01-16T09:00:00,False
3,92500305009592023,13864377000130-1-002003/2023,2023,2003,13864377000130,NaN,65479,FUNDO MUNICIPAL DE SAUDE - FMS,NaN,M,...,Edital,Aberto-Fechado,12720.30,10614.69,2024-01-02T07:01:26,2024-01-02T07:01:26,2024-01-02T07:01:26,2024-01-02T08:00:00,2024-01-29T09:00:00,False
4,38921905000122023,44407989000128-1-000001/2023,2023,1,44407989000128,NaN,82542,CONSELHO REGIONAL DE NUTRICIONISTAS 3 REGIAO,NaN,F,...,Edital,Aberto,1184364.44,1009985.64,2024-01-02T07:01:35,2024-01-02T07:01:35,2024-01-02T07:01:35,2024-01-02T08:00:00,2024-01-16T09:00:00,False


In [ ]:
#for i in todos_registros:
    #print(i)

In [9]:


# 3. Mapeia outras colunas comuns do PNCP
mapeamento = {
    'orgaoEntidade.razaoSocial': 'orgaoEntidadeRazaoSocial',
    'modalidadeNome': 'modalidadeNome',
    'numeroCompra': 'numeroCompra',
    'objetoCompra': 'objetoCompra',
    'valorTotalEstimado': 'valorTotalEstimado',
    'valorTotalHomologado': 'valorTotalHomologado',
}
df_24 = df_24.rename(
    columns={k: v for k, v in mapeamento.items() if k in df_24.columns}
)

df_24["ano"] = 2024


# 4. Trata e padroniza a coluna SRP (identifica True, 1, 'True', 'S', etc.)
if 'srp' in df_24.columns:
    # Mostra os valores brutos que vieram da API antes de converter
    print("\nValores brutos encontrados na coluna SRP:")
    print(df_24['srp'].value_counts(dropna=False))

    # Converte para booleano real
    df_24['srp_bool'] = df_24['srp'].astype(str).str.lower().isin(['true', '1', 's', 'sim'])

    df_24_com_srp = df_24[df_24['srp_bool'] == True]
    df_24_sem_srp = df_24[df_24['srp_bool'] == False]

    print(f"\n Total COM SRP: {len(df_24_com_srp)}")
    print(f" Total SEM SRP: {len(df_24_sem_srp)}")



Valores brutos encontrados na coluna SRP:
srp
False    61852
True      8049
Name: count, dtype: int64

 Total COM SRP: 8049
 Total SEM SRP: 61852


In [10]:
print("\nDataFrame com SRP:")
display(df_24_com_srp.head())
print("\nDataFrame sem SRP:")
display(df_24_sem_srp.head())


DataFrame com SRP:


,idCompra,numeroControlePNCP,anoCompraPncp,sequencialCompraPncp,orgaoEntidadeCnpj,orgaoSubrogadoCnpj,codigoOrgao,orgaoEntidadeRazaoSocial,orgaoSubrogadoRazaoSocial,orgaoEntidadeEsferaId,...,valorTotalEstimado,valorTotalHomologado,dataInclusaoPncp,dataAtualizacaoPncp,dataPublicacaoPncp,dataAberturaPropostaPncp,dataEncerramentoPropostaPncp,contratacaoExcluida,ano,srp_bool
0,17013105000172023,00394460000141-1-001631/2023,2023,1631,00394460000141,NaN,45178,MINISTERIO DA FAZENDA,NaN,F,...,614963.83,343793.5,2024-01-02T07:00:02,2024-01-02T07:00:02,2024-01-02T07:00:02,2024-01-02T09:00:00,2024-01-22T10:00:00,False,2024,True
5,92500305009532023,13864377000130-1-002004/2023,2023,2004,13864377000130,NaN,65479,FUNDO MUNICIPAL DE SAUDE - FMS,NaN,M,...,12215678.10,5134100.0,2024-01-03T07:00:17,2024-01-03T07:00:17,2024-01-03T07:00:17,2024-01-03T08:00:00,2024-01-15T09:00:00,False,2024,True
9,16051805000412023,00394452000103-1-014530/2023,2023,14530,00394452000103,NaN,44611,COMANDO DO EXERCITO,NaN,F,...,2421217.95,1301835.0,2024-01-04T07:00:28,2024-01-04T07:00:28,2024-01-04T07:00:28,2024-01-05T09:00:00,2024-01-17T10:00:00,False,2024,True
13,15303105001522023,60453032000174-1-000263/2023,2023,263,60453032000174,NaN,84162,UNIVERSIDADE FEDERAL DE SAO PAULO,NaN,F,...,17398.80,7049.9,2024-01-05T07:01:04,2024-01-11T07:04:47,2024-01-05T07:01:04,2024-01-11T08:00:00,2024-01-23T09:00:00,False,2024,True
14,38929705000462023,62655246000159-1-000099/2023,2023,99,62655246000159,NaN,37757,CONSELHO REGIONAL DE CORRETORES DE IMOVEIS DA ...,NaN,F,...,752395.33,337500.0,2024-01-05T07:01:15,2024-01-05T07:01:15,2024-01-05T07:01:15,2024-01-05T08:30:00,2024-01-17T10:00:00,False,2024,True



DataFrame sem SRP:


,idCompra,numeroControlePNCP,anoCompraPncp,sequencialCompraPncp,orgaoEntidadeCnpj,orgaoSubrogadoCnpj,codigoOrgao,orgaoEntidadeRazaoSocial,orgaoSubrogadoRazaoSocial,orgaoEntidadeEsferaId,...,valorTotalEstimado,valorTotalHomologado,dataInclusaoPncp,dataAtualizacaoPncp,dataPublicacaoPncp,dataAberturaPropostaPncp,dataEncerramentoPropostaPncp,contratacaoExcluida,ano,srp_bool
1,12006005000362023,00394429000100-1-002260/2023,2023,2260,00394429000100,NaN,43849,COMANDO DA AERONAUTICA,NaN,F,...,286011.82,2.700000e+05,2024-01-02T07:00:07,2024-01-02T07:00:07,2024-01-02T07:00:07,2024-01-02T08:00:00,2024-01-17T09:00:00,False,2024,False
2,51017805000112023,29979036000140-1-000084/2023,2023,84,29979036000140,NaN,26661,INSTITUTO NACIONAL DO SEGURO SOCIAL,NaN,F,...,232490.96,NaN,2024-01-02T07:00:09,2024-02-28T07:09:02,2024-01-02T07:00:09,2024-01-02T09:00:00,2024-01-16T09:00:00,False,2024,False
3,92500305009592023,13864377000130-1-002003/2023,2023,2003,13864377000130,NaN,65479,FUNDO MUNICIPAL DE SAUDE - FMS,NaN,M,...,12720.30,1.061469e+04,2024-01-02T07:01:26,2024-01-02T07:01:26,2024-01-02T07:01:26,2024-01-02T08:00:00,2024-01-29T09:00:00,False,2024,False
4,38921905000122023,44407989000128-1-000001/2023,2023,1,44407989000128,NaN,82542,CONSELHO REGIONAL DE NUTRICIONISTAS 3 REGIAO,NaN,F,...,1184364.44,1.009986e+06,2024-01-02T07:01:35,2024-01-02T07:01:35,2024-01-02T07:01:35,2024-01-02T08:00:00,2024-01-16T09:00:00,False,2024,False
6,07001805001422023,00509018000113-1-002153/2023,2023,2153,00509018000113,NaN,47412,TRIBUNAL SUPERIOR ELEITORAL,NaN,F,...,363660.05,3.617351e+05,2024-01-03T07:00:30,2024-01-03T07:00:30,2024-01-03T07:00:30,2024-01-03T08:00:00,2024-01-15T13:00:00,False,2024,False


In [11]:
# GroupBy agregando com nunique (únicos) e count (total de registros)
df_qnt_modalidade_orgao_24 = df_24.groupby('srp').agg(
    qtd_modalidades=('modalidadeNome', 'nunique'),
    qtd_orgaos=('orgaoEntidadeRazaoSocial', 'nunique'),
    total_registros=('numeroCompra', 'count')
)

# Renomeia os índices para facilitar a leitura no relatório
df_qnt_modalidade_orgao_24.index = ['Sem SRP', 'Com SRP']
display(df_qnt_modalidade_orgao_24)

,qtd_modalidades,qtd_orgaos,total_registros
Sem SRP,3,356,61852
Com SRP,3,150,8049


# Pegando dados 2025

In [12]:
modalidades = [5, 6] # Pegar todas as modalidades de licitação, mas para fins de teste, vamos pegar apenas as principais.
todos_registros = []

In [13]:
for modalidade in modalidades:
    pagina = 1
    total_modalidade = 0

    while True:
        params = {
            "pagina": pagina,
            "tamanhoPagina": 500,
            "dataPublicacaoPncpInicial": "2024-01-01",
            "dataPublicacaoPncpFinal": "2024-12-31",
            "codigoModalidade": modalidade,
            "unidadeOrgaoUfSigla": "SP"
        }

        resposta = requests.get(url, params=params, timeout=60)

        if resposta.status_code == 429:
            espera = 5
            print(f"Modalidade {modalidade}, página {pagina}: 429, aguardando {espera}s e tentando de novo...")
            time.sleep(espera)
            continue  # tenta a mesma página de novo, sem avançar

        if resposta.status_code != 200:
            print(f"Modalidade {modalidade}, página {pagina}: erro {resposta.status_code}")
            break

        dados = resposta.json()
        registros = extrair_registros(dados)

        if not registros:
            break

        todos_registros.extend(registros)
        total_modalidade += len(registros)

        if len(registros) < 500:
            break

        pagina += 1
        time.sleep(0.2)  # pausa maior entre páginas

    print(f"Modalidade {modalidade}: {total_modalidade} registros coletados.")
    time.sleep(3)  # pausa entre modalidades, para o servidor "esfriar"

Modalidade 5: 33327 registros coletados.
Modalidade 6: 36074 registros coletados.


In [14]:
# print("JSON retornado:", registros)
df_25 = pd.json_normalize(todos_registros)
df_25.head()

,idCompra,numeroControlePNCP,anoCompraPncp,sequencialCompraPncp,orgaoEntidadeCnpj,orgaoSubrogadoCnpj,codigoOrgao,orgaoEntidadeRazaoSocial,orgaoSubrogadoRazaoSocial,orgaoEntidadeEsferaId,...,tipoInstrumentoConvocatorioNome,modoDisputaNomePncp,valorTotalEstimado,valorTotalHomologado,dataInclusaoPncp,dataAtualizacaoPncp,dataPublicacaoPncp,dataAberturaPropostaPncp,dataEncerramentoPropostaPncp,contratacaoExcluida
0,17013105000172023,00394460000141-1-001631/2023,2023,1631,00394460000141,NaN,45178,MINISTERIO DA FAZENDA,NaN,F,...,Edital,Aberto-Fechado,614963.83,343793.50,2024-01-02T07:00:02,2024-01-02T07:00:02,2024-01-02T07:00:02,2024-01-02T09:00:00,2024-01-22T10:00:00,False
1,12006005000362023,00394429000100-1-002260/2023,2023,2260,00394429000100,NaN,43849,COMANDO DA AERONAUTICA,NaN,F,...,Edital,Aberto,286011.82,270000.00,2024-01-02T07:00:07,2024-01-02T07:00:07,2024-01-02T07:00:07,2024-01-02T08:00:00,2024-01-17T09:00:00,False
2,51017805000112023,29979036000140-1-000084/2023,2023,84,29979036000140,NaN,26661,INSTITUTO NACIONAL DO SEGURO SOCIAL,NaN,F,...,Edital,Aberto,232490.96,NaN,2024-01-02T07:00:09,2024-02-28T07:09:02,2024-01-02T07:00:09,2024-01-02T09:00:00,2024-01-16T09:00:00,False
3,92500305009592023,13864377000130-1-002003/2023,2023,2003,13864377000130,NaN,65479,FUNDO MUNICIPAL DE SAUDE - FMS,NaN,M,...,Edital,Aberto-Fechado,12720.30,10614.69,2024-01-02T07:01:26,2024-01-02T07:01:26,2024-01-02T07:01:26,2024-01-02T08:00:00,2024-01-29T09:00:00,False
4,38921905000122023,44407989000128-1-000001/2023,2023,1,44407989000128,NaN,82542,CONSELHO REGIONAL DE NUTRICIONISTAS 3 REGIAO,NaN,F,...,Edital,Aberto,1184364.44,1009985.64,2024-01-02T07:01:35,2024-01-02T07:01:35,2024-01-02T07:01:35,2024-01-02T08:00:00,2024-01-16T09:00:00,False


In [15]:
# 3. Mapeia outras colunas comuns do PNCP
mapeamento = {
    'orgaoEntidade.razaoSocial': 'orgaoEntidadeRazaoSocial',
    'modalidadeNome': 'modalidadeNome',
    'numeroCompra': 'numeroCompra',
    'objetoCompra': 'objetoCompra',
    'valorTotalEstimado': 'valorTotalEstimado',
    'valorTotalHomologado': 'valorTotalHomologado',
}
df_25 = df_25.rename(
    columns={k: v for k, v in mapeamento.items() if k in df_25.columns}
)

df_25["ano"] = 2025

# 4. Trata e padroniza a coluna SRP (identifica True, 1, 'True', 'S', etc.)
if 'srp' in df_25.columns:
    # Mostra os valores brutos que vieram da API antes de converter
    print("\nValores brutos encontrados na coluna SRP:")
    print(df_25['srp'].value_counts(dropna=False))

    # Converte para booleano real
    df_25['srp_bool'] = df_25['srp'].astype(str).str.lower().isin(['true', '1', 's', 'sim'])

    df_25_com_srp = df_25[df_25['srp_bool'] == True]
    df_25_sem_srp = df_25[df_25['srp_bool'] == False]

    print(f"\n Total COM SRP: {len(df_25_com_srp)}")
    print(f" Total SEM SRP: {len(df_25_sem_srp)}")



Valores brutos encontrados na coluna SRP:
srp
False    61551
True      7850
Name: count, dtype: int64

 Total COM SRP: 7850
 Total SEM SRP: 61551


In [16]:
print("\nDataFrame com SRP:")
display(df_25_com_srp.head())
print("\nDataFrame sem SRP:")
display(df_25_sem_srp.head())


DataFrame com SRP:


,idCompra,numeroControlePNCP,anoCompraPncp,sequencialCompraPncp,orgaoEntidadeCnpj,orgaoSubrogadoCnpj,codigoOrgao,orgaoEntidadeRazaoSocial,orgaoSubrogadoRazaoSocial,orgaoEntidadeEsferaId,...,valorTotalEstimado,valorTotalHomologado,dataInclusaoPncp,dataAtualizacaoPncp,dataPublicacaoPncp,dataAberturaPropostaPncp,dataEncerramentoPropostaPncp,contratacaoExcluida,ano,srp_bool
0,17013105000172023,00394460000141-1-001631/2023,2023,1631,00394460000141,NaN,45178,MINISTERIO DA FAZENDA,NaN,F,...,614963.83,343793.5,2024-01-02T07:00:02,2024-01-02T07:00:02,2024-01-02T07:00:02,2024-01-02T09:00:00,2024-01-22T10:00:00,False,2025,True
5,92500305009532023,13864377000130-1-002004/2023,2023,2004,13864377000130,NaN,65479,FUNDO MUNICIPAL DE SAUDE - FMS,NaN,M,...,12215678.10,5134100.0,2024-01-03T07:00:17,2024-01-03T07:00:17,2024-01-03T07:00:17,2024-01-03T08:00:00,2024-01-15T09:00:00,False,2025,True
9,16051805000412023,00394452000103-1-014530/2023,2023,14530,00394452000103,NaN,44611,COMANDO DO EXERCITO,NaN,F,...,2421217.95,1301835.0,2024-01-04T07:00:28,2024-01-04T07:00:28,2024-01-04T07:00:28,2024-01-05T09:00:00,2024-01-17T10:00:00,False,2025,True
13,15303105001522023,60453032000174-1-000263/2023,2023,263,60453032000174,NaN,84162,UNIVERSIDADE FEDERAL DE SAO PAULO,NaN,F,...,17398.80,7049.9,2024-01-05T07:01:04,2024-01-11T07:04:47,2024-01-05T07:01:04,2024-01-11T08:00:00,2024-01-23T09:00:00,False,2025,True
14,38929705000462023,62655246000159-1-000099/2023,2023,99,62655246000159,NaN,37757,CONSELHO REGIONAL DE CORRETORES DE IMOVEIS DA ...,NaN,F,...,752395.33,337500.0,2024-01-05T07:01:15,2024-01-05T07:01:15,2024-01-05T07:01:15,2024-01-05T08:30:00,2024-01-17T10:00:00,False,2025,True



DataFrame sem SRP:


,idCompra,numeroControlePNCP,anoCompraPncp,sequencialCompraPncp,orgaoEntidadeCnpj,orgaoSubrogadoCnpj,codigoOrgao,orgaoEntidadeRazaoSocial,orgaoSubrogadoRazaoSocial,orgaoEntidadeEsferaId,...,valorTotalEstimado,valorTotalHomologado,dataInclusaoPncp,dataAtualizacaoPncp,dataPublicacaoPncp,dataAberturaPropostaPncp,dataEncerramentoPropostaPncp,contratacaoExcluida,ano,srp_bool
1,12006005000362023,00394429000100-1-002260/2023,2023,2260,00394429000100,NaN,43849,COMANDO DA AERONAUTICA,NaN,F,...,286011.82,2.700000e+05,2024-01-02T07:00:07,2024-01-02T07:00:07,2024-01-02T07:00:07,2024-01-02T08:00:00,2024-01-17T09:00:00,False,2025,False
2,51017805000112023,29979036000140-1-000084/2023,2023,84,29979036000140,NaN,26661,INSTITUTO NACIONAL DO SEGURO SOCIAL,NaN,F,...,232490.96,NaN,2024-01-02T07:00:09,2024-02-28T07:09:02,2024-01-02T07:00:09,2024-01-02T09:00:00,2024-01-16T09:00:00,False,2025,False
3,92500305009592023,13864377000130-1-002003/2023,2023,2003,13864377000130,NaN,65479,FUNDO MUNICIPAL DE SAUDE - FMS,NaN,M,...,12720.30,1.061469e+04,2024-01-02T07:01:26,2024-01-02T07:01:26,2024-01-02T07:01:26,2024-01-02T08:00:00,2024-01-29T09:00:00,False,2025,False
4,38921905000122023,44407989000128-1-000001/2023,2023,1,44407989000128,NaN,82542,CONSELHO REGIONAL DE NUTRICIONISTAS 3 REGIAO,NaN,F,...,1184364.44,1.009986e+06,2024-01-02T07:01:35,2024-01-02T07:01:35,2024-01-02T07:01:35,2024-01-02T08:00:00,2024-01-16T09:00:00,False,2025,False
6,07001805001422023,00509018000113-1-002153/2023,2023,2153,00509018000113,NaN,47412,TRIBUNAL SUPERIOR ELEITORAL,NaN,F,...,363660.05,3.617351e+05,2024-01-03T07:00:30,2024-01-03T07:00:30,2024-01-03T07:00:30,2024-01-03T08:00:00,2024-01-15T13:00:00,False,2025,False


In [17]:
# GroupBy agregando com nunique (únicos) e count (total de registros)
df_qnt_modalidade_orgao_25 = df_25.groupby('srp').agg(
    qtd_modalidades=('modalidadeNome', 'nunique'),
    qtd_orgaos=('orgaoEntidadeRazaoSocial', 'nunique'),
    total_registros=('numeroCompra', 'count')
)

# Renomeia os índices para facilitar a leitura no relatório
df_qnt_modalidade_orgao_25.index = ['Sem SRP', 'Com SRP']
display(df_qnt_modalidade_orgao_25)

,qtd_modalidades,qtd_orgaos,total_registros
Sem SRP,3,356,61551
Com SRP,3,150,7850


# Comparação 2024 X 2025

In [18]:
display(df_qnt_modalidade_orgao_24)
display(df_qnt_modalidade_orgao_25)

,qtd_modalidades,qtd_orgaos,total_registros
Sem SRP,3,356,61852
Com SRP,3,150,8049


,qtd_modalidades,qtd_orgaos,total_registros
Sem SRP,3,356,61551
Com SRP,3,150,7850


In [19]:
df_total = pd.concat([df_24, df_25], ignore_index=True)
df_total

,idCompra,numeroControlePNCP,anoCompraPncp,sequencialCompraPncp,orgaoEntidadeCnpj,orgaoSubrogadoCnpj,codigoOrgao,orgaoEntidadeRazaoSocial,orgaoSubrogadoRazaoSocial,orgaoEntidadeEsferaId,...,valorTotalEstimado,valorTotalHomologado,dataInclusaoPncp,dataAtualizacaoPncp,dataPublicacaoPncp,dataAberturaPropostaPncp,dataEncerramentoPropostaPncp,contratacaoExcluida,ano,srp_bool
0,17013105000172023,00394460000141-1-001631/2023,2023,1631,00394460000141,NaN,45178,MINISTERIO DA FAZENDA,NaN,F,...,614963.83,343793.50,2024-01-02T07:00:02,2024-01-02T07:00:02,2024-01-02T07:00:02,2024-01-02T09:00:00,2024-01-22T10:00:00,False,2024,True
1,12006005000362023,00394429000100-1-002260/2023,2023,2260,00394429000100,NaN,43849,COMANDO DA AERONAUTICA,NaN,F,...,286011.82,270000.00,2024-01-02T07:00:07,2024-01-02T07:00:07,2024-01-02T07:00:07,2024-01-02T08:00:00,2024-01-17T09:00:00,False,2024,False
2,51017805000112023,29979036000140-1-000084/2023,2023,84,29979036000140,NaN,26661,INSTITUTO NACIONAL DO SEGURO SOCIAL,NaN,F,...,232490.96,NaN,2024-01-02T07:00:09,2024-02-28T07:09:02,2024-01-02T07:00:09,2024-01-02T09:00:00,2024-01-16T09:00:00,False,2024,False
3,92500305009592023,13864377000130-1-002003/2023,2023,2003,13864377000130,NaN,65479,FUNDO MUNICIPAL DE SAUDE - FMS,NaN,M,...,12720.30,10614.69,2024-01-02T07:01:26,2024-01-02T07:01:26,2024-01-02T07:01:26,2024-01-02T08:00:00,2024-01-29T09:00:00,False,2024,False
4,38921905000122023,44407989000128-1-000001/2023,2023,1,44407989000128,NaN,82542,CONSELHO REGIONAL DE NUTRICIONISTAS 3 REGIAO,NaN,F,...,1184364.44,1009985.64,2024-01-02T07:01:35,2024-01-02T07:01:35,2024-01-02T07:01:35,2024-01-02T08:00:00,2024-01-16T09:00:00,False,2024,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
139297,08032806000092024,46384111000140-1-001605/2024,2024,1605,46384111000140,NaN,35411,SAO PAULO SECRETARIA DA EDUCACAO,NaN,E,...,1162.00,1162.00,2024-12-30T17:09:45,2024-12-30T17:09:45,2024-12-30T17:09:45,NaN,NaN,False,2025,False
139298,09020306001042024,46374500000194-1-011338/2024,2024,11338,46374500000194,NaN,34952,SECRETARIA DE ESTADO DA SAUDE,NaN,E,...,7740.00,7740.00,2024-12-30T17:15:31,2024-12-30T17:15:31,2024-12-30T17:15:31,NaN,NaN,False,2025,False
139299,09017706000792024,46374500000194-1-011339/2024,2024,11339,46374500000194,NaN,34952,SECRETARIA DE ESTADO DA SAUDE,NaN,E,...,9120.00,7710.00,2024-12-30T18:56:04,2025-01-31T10:32:31,2024-12-30T18:56:04,NaN,NaN,False,2025,False
139300,09002906001192024,59949362000176-1-000160/2024,2024,160,59949362000176,NaN,37676,TRIBUNAL REGIONAL FEDERAL 3 REGIAO,NaN,F,...,13842.40,13842.40,2024-12-30T19:24:51,2024-12-30T19:37:42,2024-12-30T19:24:51,NaN,NaN,False,2025,False


In [20]:
# 1. Lista com o nome exato das colunas de maior relevância
colunas_principais = [
    'srp',
    'ano',
    'numeroCompra',
    'modalidadeNome',
    'orgaoEntidadeRazaoSocial',
    'objetoCompra',
    'valorTotalEstimado',
    'valorTotalHomologado',
    'dataPublicacaoPncp',
]

# 2. Seleciona apenas as colunas que realmente existem no df_total
colunas_presentes = [c for c in colunas_principais if c in df_total.columns]
df_final_reduzido = df_total[colunas_presentes].copy()

# 3. Tratamento de Tipos de Dados


# B) Converte colunas financeiras para numérico (Float)
for col_valor in ['valorTotalEstimado', 'valorTotalHomologado']:
    if col_valor in df_final_reduzido.columns:
        df_final_reduzido[col_valor] = pd.to_numeric(
            df_final_reduzido[col_valor], errors='coerce'
        )

# C) Converte a data de publicação para formato de data real
if 'dataPublicacaoPncp' in df_final_reduzido.columns:
    df_final_reduzido['dataPublicacaoPncp'] = pd.to_datetime(
        df_final_reduzido['dataPublicacaoPncp'], errors='coerce'
    )

print("=== DATAFRAME FINAL REDUZIDO ===")
print(f"Dimensões do DataFrame: {df_final_reduzido.shape}")
display(df_final_reduzido.head(10))

=== DATAFRAME FINAL REDUZIDO ===
Dimensões do DataFrame: (139302, 9)


,srp,ano,numeroCompra,modalidadeNome,orgaoEntidadeRazaoSocial,objetoCompra,valorTotalEstimado,valorTotalHomologado,dataPublicacaoPncp
0,True,2024,00017,Pregão - Eletrônico,MINISTERIO DA FAZENDA,"Contratação, através de Ata de Registro de Pre...",614963.83,3.437935e+05,2024-01-02 07:00:02
1,False,2024,00036,Pregão - Eletrônico,COMANDO DA AERONAUTICA,O objeto da presente licitação é a Contratação...,286011.82,2.700000e+05,2024-01-02 07:00:07
2,False,2024,00011,Pregão - Eletrônico,INSTITUTO NACIONAL DO SEGURO SOCIAL,Contratação dos serviços de administração e ge...,232490.96,NaN,2024-01-02 07:00:09
3,False,2024,00959,Pregão - Eletrônico,FUNDO MUNICIPAL DE SAUDE - FMS,"AQUISIÇÃO DE FERRAMENTAS DIVERSAS, conforme es...",12720.30,1.061469e+04,2024-01-02 07:01:26
4,False,2024,00012,Pregão - Eletrônico,CONSELHO REGIONAL DE NUTRICIONISTAS 3 REGIAO,Serviços de assistência médica ambulatorial e ...,1184364.44,1.009986e+06,2024-01-02 07:01:35
5,True,2024,00953,Pregão - Eletrônico,FUNDO MUNICIPAL DE SAUDE - FMS,REGISTRO De PREÇOS OBJETIVANDO O FORNECIMENTO ...,12215678.10,5.134100e+06,2024-01-03 07:00:17
6,False,2024,00142,Pregão - Eletrônico,TRIBUNAL SUPERIOR ELEITORAL,Fornecimento parcelado de combustíveis (gasoli...,363660.05,3.617351e+05,2024-01-03 07:00:30
7,False,2024,91710,Pregão - Eletrônico,"INSTITUTO FEDERAL DE EDUCACAO, CIENCIA E TECNO...",Contratação de serviço de outsourcing de impre...,74400.00,7.186560e+04,2024-01-03 07:00:44
8,False,2024,01154,Pregão - Eletrônico,SERVICO FEDERAL DE PROCESSAMENTO DE DADOS (SER...,Serviço de Manutenção e suporte técnico de Sub...,229320.00,4.752000e+04,2024-01-04 07:00:10
9,True,2024,00041,Pregão - Eletrônico,COMANDO DO EXERCITO,Aquisição de material de informática,2421217.95,1.301835e+06,2024-01-04 07:00:28


In [21]:
df_final_reduzido.to_parquet("srp_contratacoes_tratado.parquet", index=False)

In [ ]:
df_final_reduzido = pd.read_parquet("srp_contratacoes_tratado.parquet")